# Ball Tracking + Possession Assignment + Annotation

Three stages, run top to bottom:

1. **Ball tracker** — implausible-detection pre-filter (confidence floor + physically-expected ball size from homography + player head/torso containment veto) removes false positives *before* they ever reach the trajectory logic, then physically-grounded outlier rejection (real speed via homography, not raw pixels) + bounded gap interpolation + short-window Kalman smoothing (jitter only, resets on every gap so it never coasts through a stop). Produces `ball_tracked_cache`.
2. **Possession assigner** — sticky/hysteresis nearest-player-to-ball in pitch-meters, fed by the *smoothed* ball position from stage 1. A carrier only changes hands when a different player is clearly closer for several consecutive frames. Produces `ball_carrier_cache`.
3. **Annotation** — foot-ellipse + floating ID badge above the head (never covers the player), team-colored, referee in yellow, plus a highlighted ring on the current ball carrier. Renders a preview video.

**Fix (this version):** the tracker was previously handing every raw per-frame detection straight into outlier rejection with no confidence or plausibility check. Outlier rejection only catches unrealistic *speed jumps*, so a low-confidence false positive sitting still on a player's head/torso (moving at player speed, not ball speed) sailed straight through and got smoothed into a stable wrong lock. The new `filter_implausible_detections` step rejects those before they ever reach the trajectory logic.


In [ ]:
import pickle
import cv2
import numpy as np
from collections import Counter

from paths import (
    VIDEO_PATH,
    TRACKING_CACHE_PATH,
    HOMOGRAPHY_CACHE_PATH,
    TEAM_CACHE_PATH,
    BALL_TRACKED_CACHE_PATH,
    BALL_CARRIER_CACHE_PATH,
    BALL_CARRIER_PREVIEW_VIDEO_PATH,
)

PX_PER_METER = 10   # must match your homography engine's config

# ---- ball detection pre-filter (NEW) ----
BALL_DIAMETER_M = 0.22
MIN_BALL_CONF = 0.30          # tune against your detector's actual conf distribution
MAX_SIZE_RATIO = 2.4          # detected bbox this many x larger than expected physical ball size -> reject
MIN_SIZE_RATIO = 0.35         # this many x smaller -> reject (noise)
HEAD_CONTAINMENT_THRESH = 0.75
HEAD_REGION_FRAC = 0.4        # top 40% of a player's bbox height counts as head/torso zone

# ---- ball tracker ----
MAX_BALL_SPEED_KMH = 130.0   # realistic max struck-ball speed; anything above -> mis-detection
MAX_GAP_INTERP = 30           # frames; bounded linear interpolation across a ball-detection gap
SMOOTH_PROCESS_NOISE = 5e-2   # kalman process noise (jitter smoothing only, not long-range prediction)
SMOOTH_MEASUREMENT_NOISE = 5e-1

# ---- possession assigner ----
MAX_POSSESSION_DIST_M = 2.0   # a player must be within this many meters of the ball to be considered at all
SWITCH_MARGIN_M = 0.5         # a new candidate must be closer than the current carrier by at least this much...
CONFIRM_FRAMES = 5            # ...for this many consecutive frames before possession actually switches
EXCLUDE_ROLES = {"referee"}   # goalkeepers ARE eligible; referees never are

print("Config set.")


In [2]:
with open(TRACKING_CACHE_PATH, "rb") as f:
    tracking_data = pickle.load(f)

tracking_cache = tracking_data["tracking_cache"]          # {frame_idx: {"ball": det_or_None, "tracks": [...]}}
locked_class_by_id = tracking_data["locked_class_by_id"]  # {track_id: "player"/"goalkeeper"/"referee"}

with open(HOMOGRAPHY_CACHE_PATH, "rb") as f:
    homography_cache = pickle.load(f)   # list[H or None], index = frame_idx

with open(TEAM_CACHE_PATH, "rb") as f:
    team_data = pickle.load(f)

team_by_id = team_data["team_by_id"]                              # {track_id: 0 or 1}
goalkeeper_team_by_id = team_data["goalkeeper_team_assignment"]   # {track_id: 0 or 1}

total_frames = max(tracking_cache.keys()) + 1

cap_probe = cv2.VideoCapture(VIDEO_PATH)
FPS = cap_probe.get(cv2.CAP_PROP_FPS)
cap_probe.release()

print(f"Tracking cache: {len(tracking_cache)} frames")
print(f"Locked classes: {len(locked_class_by_id)} tracks")
print(f"Homography cache: {len(homography_cache)} entries")
print(f"Team assignment: {len(team_by_id)} tracks")
print(f"Video FPS: {FPS:.2f}")


Tracking cache: 3001 frames
Locked classes: 25 tracks
Homography cache: 3001 entries
Team assignment: 22 tracks
Video FPS: 25.00


## Ball tracker — helpers

In [ ]:
def get_homography_at(homography_cache, frame_idx):
    if isinstance(homography_cache, dict):
        return homography_cache.get(frame_idx)
    if 0 <= frame_idx < len(homography_cache):
        return homography_cache[frame_idx]
    return None


def pixel_to_pitch(H, px, py, px_per_meter=PX_PER_METER):
    pt = cv2.perspectiveTransform(
        np.array([[[px, py]]], dtype=np.float32), H
    ).reshape(2)
    return float(pt[0] / px_per_meter), float(pt[1] / px_per_meter)


def bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def bbox_foot_point(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, y2)


def reject_outliers_metric(centers, homography_cache, fps, max_speed_kmh):
    """Physically-grounded outlier rejection: reject a detection if the
    implied real-world ball speed between consecutive frames exceeds a
    realistic max, computed via homography (meters), not raw pixels."""
    max_speed_mps = max_speed_kmh / 3.6
    max_dist_per_frame = max_speed_mps / fps

    cleaned = list(centers)
    for i in range(1, len(cleaned) - 1):
        c, prev_c, next_c = cleaned[i], cleaned[i - 1], cleaned[i + 1]
        if c is None or prev_c is None:
            continue

        H_i, H_prev = get_homography_at(homography_cache, i), get_homography_at(homography_cache, i - 1)
        if H_i is None or H_prev is None:
            continue

        px1, py1 = pixel_to_pitch(H_prev, *prev_c)
        px2, py2 = pixel_to_pitch(H_i, *c)
        dist_m = np.hypot(px2 - px1, py2 - py1)

        if dist_m > max_dist_per_frame:
            if next_c is not None:
                H_next = get_homography_at(homography_cache, i + 1)
                if H_next is not None:
                    px3, py3 = pixel_to_pitch(H_next, *next_c)
                    dist_next_prev = np.hypot(px3 - px1, py3 - py1)
                    if dist_next_prev < dist_m:
                        cleaned[i] = None
    return cleaned


def interpolate_gaps(centers, max_gap):
    """Bounded linear interpolation between two confirmed detections --
    never extrapolates past a real gap, so a reception/stop is handled
    correctly (interpolation lands where the receiver actually is, not
    wherever momentum would have carried the ball)."""
    result = list(centers)
    sources = ["detected" if c is not None else None for c in centers]
    i = 0
    while i < len(result):
        if result[i] is None:
            j = i
            while j < len(result) and result[j] is None:
                j += 1
            gap_len = j - i
            has_before = i > 0 and result[i - 1] is not None
            has_after = j < len(result) and result[j] is not None
            if has_before and has_after and gap_len <= max_gap:
                x0, y0 = result[i - 1]
                x1, y1 = result[j]
                for k in range(gap_len):
                    t = (k + 1) / (gap_len + 1)
                    result[i + k] = (x0 + (x1 - x0) * t, y0 + (y1 - y0) * t)
                    sources[i + k] = "interpolated"
            i = j
        else:
            i += 1
    return result, sources


def smooth_kalman(centers, sources, process_noise, measurement_noise):
    """Short-window jitter smoother only -- resets on every gap so it never
    coasts/predicts through missing data, only smooths within continuous
    detection runs."""
    smoothed = list(centers)
    kf = cv2.KalmanFilter(4, 2)
    kf.measurementMatrix = np.array([[1,0,0,0],[0,1,0,0]], dtype=np.float32)
    kf.transitionMatrix = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], dtype=np.float32)
    kf.processNoiseCov = np.eye(4, dtype=np.float32) * process_noise
    kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * measurement_noise
    kf.errorCovPost = np.eye(4, dtype=np.float32)

    initialized = False
    for i in range(len(centers)):
        c = centers[i]
        if c is None:
            initialized = False
            continue
        if not initialized:
            kf.statePost = np.array([[c[0]], [c[1]], [0], [0]], dtype=np.float32)
            initialized = True
            smoothed[i] = c
            continue
        kf.predict()
        measurement = np.array([[np.float32(c[0])], [np.float32(c[1])]])
        corrected = kf.correct(measurement)
        if sources[i] == "detected":
            smoothed[i] = (float(corrected[0]), float(corrected[1]))
        else:
            smoothed[i] = c
    return smoothed


def expected_ball_pixel_size(H, cx, cy, px_per_meter=PX_PER_METER, ball_diameter_m=BALL_DIAMETER_M):
    """Physically-grounded expected ball diameter in pixels at this image
    location, derived from the local scale of the homography -- same
    principle as the speed-based outlier check above, applied to size
    instead of motion."""
    if H is None:
        return None
    try:
        H_inv = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        return None
    pitch_pt = cv2.perspectiveTransform(np.array([[[cx, cy]]], dtype=np.float32), H).reshape(2)
    offset = pitch_pt + np.array([ball_diameter_m * px_per_meter, 0], dtype=np.float32)
    back = cv2.perspectiveTransform(np.array([[offset]], dtype=np.float32), H_inv).reshape(2)
    return float(np.hypot(back[0] - cx, back[1] - cy))


def bbox_containment(box_a, box_b):
    """Fraction of box_a's area that lies inside box_b."""
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(1e-6, (ax2 - ax1) * (ay2 - ay1))
    return inter / area_a


def filter_implausible_detections(tracking_cache, homography_cache, locked_class_by_id, total_frames):
    """Pre-filter raw ball detections BEFORE they reach speed-based outlier
    rejection. reject_outliers_metric only catches unrealistic frame-to-frame
    SPEED, so a low-confidence false positive that moves at plausible player
    speed (because it's glued to a player's head/torso) sails straight
    through it and gets smoothed into a stable wrong lock. This catches that
    failure mode directly via confidence, physically-expected ball size, and
    containment inside a player's head/torso region."""
    centers = [None] * total_frames
    reasons = [None] * total_frames

    for f in range(total_frames):
        ball = tracking_cache.get(f, {}).get("ball")
        if ball is None:
            continue

        bbox = ball["bbox"]
        conf = ball.get("conf", 1.0)
        cx, cy = bbox_center(bbox)

        if conf < MIN_BALL_CONF:
            reasons[f] = "low_conf"
            continue

        H = get_homography_at(homography_cache, f)
        expected_px = expected_ball_pixel_size(H, cx, cy)
        if expected_px and expected_px > 1e-3:
            actual_px = ((bbox[2] - bbox[0]) + (bbox[3] - bbox[1])) / 2.0
            ratio = actual_px / expected_px
            if ratio > MAX_SIZE_RATIO or ratio < MIN_SIZE_RATIO:
                reasons[f] = "implausible_size"
                continue

        rejected = False
        for t in tracking_cache.get(f, {}).get("tracks", []):
            role = locked_class_by_id.get(t["track_id"], t["class"])
            if role == "referee":
                continue
            px1, py1, px2, py2 = t["bbox"]
            head_zone = (px1, py1, px2, py1 + HEAD_REGION_FRAC * (py2 - py1))
            if bbox_containment(bbox, head_zone) >= HEAD_CONTAINMENT_THRESH:
                reasons[f] = "head_region_fp"
                rejected = True
                break
        if not rejected:
            centers[f] = (cx, cy)

    return centers, reasons


print("Ball tracker helpers ready.")


## Ball tracker — pre-filter + run

In [ ]:
raw_centers, reject_reasons = filter_implausible_detections(
    tracking_cache, homography_cache, locked_class_by_id, total_frames
)

centers_clean = reject_outliers_metric(raw_centers, homography_cache, FPS, MAX_BALL_SPEED_KMH)
centers_filled, sources = interpolate_gaps(centers_clean, MAX_GAP_INTERP)
centers_smoothed = smooth_kalman(centers_filled, sources, SMOOTH_PROCESS_NOISE, SMOOTH_MEASUREMENT_NOISE)

ball_tracked_cache = {}
for f in range(total_frames):
    c = centers_smoothed[f]
    if c is not None:
        ball_tracked_cache[f] = {"center": (float(c[0]), float(c[1])), "source": sources[f]}
    else:
        ball_tracked_cache[f] = None

detected_n = sum(1 for v in ball_tracked_cache.values() if v and v["source"] == "detected")
interp_n = sum(1 for v in ball_tracked_cache.values() if v and v["source"] == "interpolated")
none_n = sum(1 for v in ball_tracked_cache.values() if v is None)

low_conf_n = sum(1 for r in reject_reasons if r == "low_conf")
size_n = sum(1 for r in reject_reasons if r == "implausible_size")
head_fp_n = sum(1 for r in reject_reasons if r == "head_region_fp")

print("Pre-filter rejections:")
print(f"  low_conf:         {low_conf_n}")
print(f"  implausible_size: {size_n}")
print(f"  head_region_fp:   {head_fp_n}")

print(f"\nProcessed {total_frames} frames")
print(f"  detected (smoothed): {detected_n} ({100*detected_n/total_frames:.1f}%)")
print(f"  interpolated:        {interp_n} ({100*interp_n/total_frames:.1f}%)")
print(f"  none:                {none_n} ({100*none_n/total_frames:.1f}%)")

with open(BALL_TRACKED_CACHE_PATH, "wb") as f:
    pickle.dump(ball_tracked_cache, f)
print(f"\nSaved to {BALL_TRACKED_CACHE_PATH}")


## Possession assigner — candidates + sticky/hysteresis assignment

In [5]:
# for each frame: list of (track_id, distance_m) for every eligible player within
# range, sorted closest-first. None if no ball / no homography / nobody in range.
frame_candidates = [None] * total_frames

for f in range(total_frames):
    ball_entry = ball_tracked_cache.get(f)
    H = get_homography_at(homography_cache, f)

    if ball_entry is None or H is None:
        continue

    ball_pitch = pixel_to_pitch(H, *ball_entry["center"])

    candidates = []
    for t in tracking_cache.get(f, {}).get("tracks", []):
        tid = t["track_id"]
        role = locked_class_by_id.get(tid, t["class"])
        if role in EXCLUDE_ROLES:
            continue

        player_pitch = pixel_to_pitch(H, *bbox_foot_point(t["bbox"]))
        dist = float(np.hypot(player_pitch[0] - ball_pitch[0], player_pitch[1] - ball_pitch[1]))
        if dist <= MAX_POSSESSION_DIST_M:
            candidates.append((tid, dist))

    if candidates:
        candidates.sort(key=lambda c: c[1])
        frame_candidates[f] = candidates

frames_with_candidate = sum(1 for c in frame_candidates if c is not None)
print(f"Frames with at least one eligible candidate within {MAX_POSSESSION_DIST_M}m: "
      f"{frames_with_candidate}/{total_frames} ({100*frames_with_candidate/total_frames:.1f}%)")


Frames with at least one eligible candidate within 2.0m: 1252/3001 (41.7%)


In [6]:
carrier_by_frame = [None] * total_frames

current_carrier = None
pending_candidate = None
pending_count = 0

for f in range(total_frames):
    candidates = frame_candidates[f]

    if candidates is None:
        carrier_by_frame[f] = current_carrier
        pending_candidate, pending_count = None, 0
        continue

    best_id, best_dist = candidates[0]

    if current_carrier is None:
        current_carrier = best_id
        pending_candidate, pending_count = None, 0
        carrier_by_frame[f] = current_carrier
        continue

    current_dist = next((d for tid, d in candidates if tid == current_carrier), None)

    if best_id == current_carrier:
        pending_candidate, pending_count = None, 0
        carrier_by_frame[f] = current_carrier
        continue

    is_clearly_closer = (current_dist is None) or (current_dist - best_dist >= SWITCH_MARGIN_M)

    if is_clearly_closer:
        if best_id == pending_candidate:
            pending_count += 1
        else:
            pending_candidate, pending_count = best_id, 1

        if pending_count >= CONFIRM_FRAMES:
            current_carrier = pending_candidate
            pending_candidate, pending_count = None, 0
    else:
        pending_candidate, pending_count = None, 0

    carrier_by_frame[f] = current_carrier

possession_frames = sum(1 for c in carrier_by_frame if c is not None)
print(f"Frames with an assigned carrier: {possession_frames}/{total_frames} ({100*possession_frames/total_frames:.1f}%)")

counts = Counter(c for c in carrier_by_frame if c is not None)
print("\nTop possession by track_id:")
for tid, n in counts.most_common(10):
    role = locked_class_by_id.get(tid, "?")
    print(f"  id={tid:2d} ({role:10s}) | {n} frames ({100*n/total_frames:.1f}%)")

ball_carrier_cache = {f: carrier_by_frame[f] for f in range(total_frames)}
with open(BALL_CARRIER_CACHE_PATH, "wb") as f:
    pickle.dump(ball_carrier_cache, f)
print(f"\nSaved to {BALL_CARRIER_CACHE_PATH}")


Frames with an assigned carrier: 2972/3001 (99.0%)

Top possession by track_id:
  id=10 (player    ) | 634 frames (21.1%)
  id=23 (player    ) | 499 frames (16.6%)
  id=12 (player    ) | 486 frames (16.2%)
  id= 9 (player    ) | 232 frames (7.7%)
  id=16 (player    ) | 231 frames (7.7%)
  id=21 (player    ) | 136 frames (4.5%)
  id=19 (player    ) | 111 frames (3.7%)
  id=22 (player    ) | 103 frames (3.4%)
  id=17 (player    ) | 83 frames (2.8%)
  id= 4 (player    ) | 81 frames (2.7%)

Saved to barca_atletico/cache/barca_atletico_ball_carrier_cache.pkl


## Annotation — professional rendering + carrier highlight

Same layout as the project's `annotation` module (foot-ellipse + floating ID badge above the head, team-colored, referee in yellow), reproduced standalone here for the notebook, plus a highlighted ring on the current ball carrier.

In [7]:
TEAM_COLORS = {
    0: (255, 120, 0),      # BGR -- team 0, blue-ish
    1: (0, 60, 220),       # BGR -- team 1, red-ish
}
GOALKEEPER_COLOR = (0, 165, 255)     # orange
REFEREE_COLOR = (0, 230, 255)        # yellow
UNASSIGNED_COLOR = (180, 180, 180)   # gray fallback

BALL_COLOR = (0, 220, 255)
BALL_LOW_CONF_COLOR = (0, 150, 210)
CARRIER_HIGHLIGHT_COLOR = (60, 220, 60)   # bright green ring on the current carrier

FRAME_LABEL_COLOR = (0, 255, 255)
BADGE_TEXT_COLOR = (255, 255, 255)


def get_track_color(track_id, role, team_by_id, goalkeeper_team_by_id):
    if role == "referee":
        return REFEREE_COLOR
    if role == "goalkeeper":
        return GOALKEEPER_COLOR
    team = team_by_id.get(track_id)
    if team is not None:
        return TEAM_COLORS.get(team, UNASSIGNED_COLOR)
    return UNASSIGNED_COLOR


def bbox_top_point(bbox):
    x1, y1, x2, y2 = bbox
    return int((x1 + x2) / 2), int(y1)


def draw_foot_ellipse(frame, bbox, color, thickness=2):
    x1, _, x2, _ = bbox
    cx, cy = bbox_foot_point(bbox)
    width = max(int((x2 - x1) * 0.5), 12)
    cv2.ellipse(frame, (int(cx), int(cy)), (width, 6), 0, -40, 220, color, thickness, cv2.LINE_AA)


def draw_id_badge(frame, bbox, text, color, font_scale=0.42, thickness=1, gap_above_head=14):
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), _ = cv2.getTextSize(text, font, font_scale, thickness)

    top_x, top_y = bbox_top_point(bbox)
    badge_cy = top_y - gap_above_head - th // 2
    badge_cx = top_x

    pad_x, pad_y = 6, 4
    box_w = tw + 2 * pad_x
    box_h = th + 2 * pad_y
    x1 = int(badge_cx - box_w / 2)
    y1 = int(badge_cy - box_h / 2)
    x2 = x1 + box_w
    y2 = y1 + box_h
    r = box_h // 2

    cv2.line(frame, (top_x, y2), (top_x, top_y), color, 1, cv2.LINE_AA)

    cv2.rectangle(frame, (x1 + r, y1), (x2 - r, y2), color, -1, cv2.LINE_AA)
    cv2.circle(frame, (x1 + r, y1 + r), r, color, -1, cv2.LINE_AA)
    cv2.circle(frame, (x2 - r, y1 + r), r, color, -1, cv2.LINE_AA)

    cv2.putText(frame, text, (x1 + pad_x, y2 - pad_y), font, font_scale, BADGE_TEXT_COLOR, thickness, cv2.LINE_AA)


def draw_carrier_highlight(frame, bbox, color=CARRIER_HIGHLIGHT_COLOR):
    """Highlight ring around the current carrier's foot point -- drawn
    slightly larger/outside the normal foot ellipse so it doesn't collide
    with it, and never overlaps the body."""
    cx, cy = bbox_foot_point(bbox)
    x1, _, x2, _ = bbox
    width = max(int((x2 - x1) * 0.75), 18)
    cv2.ellipse(frame, (int(cx), int(cy)), (width, 10), 0, 0, 360, color, 2, cv2.LINE_AA)


def draw_dashed_circle(frame, center, radius, color, thickness=2, n_dashes=16):
    for i in range(n_dashes):
        if i % 2 == 0:
            theta1 = 360 * i / n_dashes
            theta2 = 360 * (i + 0.6) / n_dashes
            cv2.ellipse(frame, center, (radius, radius), 0, theta1, theta2, color, thickness, cv2.LINE_AA)


def draw_ball(frame, ball_entry):
    if ball_entry is None:
        return
    cx, cy = int(ball_entry["center"][0]), int(ball_entry["center"][1])
    radius = 8
    if ball_entry.get("source") == "interpolated":
        draw_dashed_circle(frame, (cx, cy), radius, BALL_LOW_CONF_COLOR, thickness=1)
    else:
        cv2.circle(frame, (cx, cy), radius, BALL_COLOR, 2, cv2.LINE_AA)
        cv2.circle(frame, (cx, cy), 1, BALL_COLOR, -1, cv2.LINE_AA)


def annotate_frame(frame, tracks, ball_entry, carrier_id, locked_class_by_id, team_by_id,
                    goalkeeper_team_by_id, show_frame_number=None):
    for t in tracks:
        tid = t["track_id"]
        role = locked_class_by_id.get(tid, t["class"])
        color = get_track_color(tid, role, team_by_id, goalkeeper_team_by_id)

        draw_foot_ellipse(frame, t["bbox"], color)

        if role == "referee":
            label = "REF"
        elif role == "goalkeeper":
            team = team_by_id.get(tid)
            label = f"GK{tid}" if team is None else f"GK{tid}-T{team}"
        else:
            team = team_by_id.get(tid)
            label = f"{tid}" if team is None else f"{tid}-T{team}"

        draw_id_badge(frame, t["bbox"], label, color)

        if tid == carrier_id:
            draw_carrier_highlight(frame, t["bbox"])

    draw_ball(frame, ball_entry)

    if show_frame_number is not None:
        cv2.putText(frame, f"frame={show_frame_number}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, FRAME_LABEL_COLOR, 2, cv2.LINE_AA)

    return frame


print("Annotation functions ready.")


Annotation functions ready.


## Render preview video

In [8]:
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Could not open video: {VIDEO_PATH}"

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

writer = cv2.VideoWriter(BALL_CARRIER_PREVIEW_VIDEO_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    data = tracking_cache.get(frame_idx, {"ball": None, "tracks": []})
    ball_entry = ball_tracked_cache.get(frame_idx)
    carrier_id = ball_carrier_cache.get(frame_idx)

    annotate_frame(
        frame,
        tracks=data.get("tracks", []),
        ball_entry=ball_entry,
        carrier_id=carrier_id,
        locked_class_by_id=locked_class_by_id,
        team_by_id=team_by_id,
        goalkeeper_team_by_id=goalkeeper_team_by_id,
    )

    writer.write(frame)
    if frame_idx % 200 == 0:
        print(f"rendered {frame_idx}/{total}")
    frame_idx += 1

cap.release()
writer.release()
print(f"\nDone. Preview saved to {BALL_CARRIER_PREVIEW_VIDEO_PATH} ({frame_idx} frames).")


rendered 0/3001
rendered 200/3001
rendered 400/3001
rendered 600/3001
rendered 800/3001
rendered 1000/3001
rendered 1200/3001
rendered 1400/3001
rendered 1600/3001
rendered 1800/3001
rendered 2000/3001
rendered 2200/3001
rendered 2400/3001
rendered 2600/3001
rendered 2800/3001
rendered 3000/3001

Done. Preview saved to barca_atletico/output/ball_carrier_preview.mkv (3001 frames).


In [10]:
# ---- DEBUG annotation v2: distance-to-ball labels + carrier highlight only ----

DEBUG_CARRIER_COLOR = (60, 220, 60)
DEBUG_CANDIDATE_COLOR = (0, 230, 255)   # yellow ring = within MAX_POSSESSION_DIST_M this frame
DEBUG_OUT_OF_RANGE_COLOR = (120, 120, 120)


def annotate_frame_debug(frame, frame_idx, tracks, ball_entry, carrier_id):
    H = get_homography_at(homography_cache, frame_idx)
    ball_pitch = pixel_to_pitch(H, *ball_entry["center"]) if (ball_entry is not None and H is not None) else None

    for t in tracks:
        tid = t["track_id"]
        fx, fy = bbox_foot_point(t["bbox"])
        fx, fy = int(fx), int(fy)

        dist_m = None
        if ball_pitch is not None and H is not None:
            player_pitch = pixel_to_pitch(H, fx, fy)
            dist_m = float(np.hypot(player_pitch[0] - ball_pitch[0], player_pitch[1] - ball_pitch[1]))

        is_carrier = (tid == carrier_id)
        in_range = dist_m is not None and dist_m <= MAX_POSSESSION_DIST_M

        ring_color = DEBUG_CARRIER_COLOR if is_carrier else (DEBUG_CANDIDATE_COLOR if in_range else DEBUG_OUT_OF_RANGE_COLOR)
        cv2.circle(frame, (fx, fy), 5, (0, 0, 0), -1, cv2.LINE_AA)
        cv2.circle(frame, (fx, fy), 7 if is_carrier else 5, ring_color, 2, cv2.LINE_AA)

        if dist_m is not None:
            label = f"{dist_m:.2f}m" + (" CARRIER" if is_carrier else "")
            cv2.putText(frame, label, (fx + 8, fy - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 3, cv2.LINE_AA)
            cv2.putText(frame, label, (fx + 8, fy - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, ring_color, 1, cv2.LINE_AA)

    if ball_entry is not None:
        bx, by = int(ball_entry["center"][0]), int(ball_entry["center"][1])
        cv2.circle(frame, (bx, by), 8, BALL_COLOR, 2, cv2.LINE_AA)

    return frame


# ---- render debug preview ----
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Could not open video: {VIDEO_PATH}"

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

DEBUG_PREVIEW_VIDEO_PATH = BALL_CARRIER_PREVIEW_VIDEO_PATH.replace(".mkv", "_debug.mkv")
writer = cv2.VideoWriter(DEBUG_PREVIEW_VIDEO_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    data = tracking_cache.get(frame_idx, {"ball": None, "tracks": []})
    ball_entry = ball_tracked_cache.get(frame_idx)
    carrier_id = ball_carrier_cache.get(frame_idx)

    annotate_frame_debug(
        frame, frame_idx,
        tracks=data.get("tracks", []),
        ball_entry=ball_entry,
        carrier_id=carrier_id,
    )

    writer.write(frame)
    if frame_idx % 200 == 0:
        print(f"rendered {frame_idx}/{total}")
    frame_idx += 1

cap.release()
writer.release()
print(f"\nDone. Debug preview saved to {DEBUG_PREVIEW_VIDEO_PATH} ({frame_idx} frames).")

rendered 0/3001
rendered 200/3001
rendered 400/3001
rendered 600/3001
rendered 800/3001
rendered 1000/3001
rendered 1200/3001
rendered 1400/3001
rendered 1600/3001
rendered 1800/3001
rendered 2000/3001
rendered 2200/3001
rendered 2400/3001
rendered 2600/3001
rendered 2800/3001
rendered 3000/3001

Done. Debug preview saved to barca_atletico/output/ball_carrier_preview_debug.mkv (3001 frames).
